#TASK #01

Which heuristic is better here (Manhattan vs Euclidean)?

Manhattan Distance is usually more suitable for 8-puzzle because tiles move only up/down/left/right (no diagonal moves), so Manhattan matches the real movement style better.

It’s admissible (never overestimates), so A* still gives the optimal shortest solution.

Manhattan usually expands fewer states than Euclidean for this puzzle, so it tends to be faster while still optimal.

In [17]:
import heapq

GOAL_STATE = ((1,2,3),
              (4,5,6),
              (7,8,0))

MOVES = [(-1,0),(1,0),(0,-1),(0,1)]


class Node:
    def __init__(self, state, parent=None, move=None, g=0):
        self.state = state
        self.parent = parent
        self.move = move
        self.g = g
        self.h = self.heuristic_function()
        self.f = self.g + self.h

    def heuristic_function(self):
        total = 0  # Manhattan Distance 
        for i in range(3):
            for j in range(3):
                val = self.state[i][j]
                if val != 0:
                    goal_row, goal_column = find_position_in_goal(val)
                    total = total + abs(i - goal_row) + abs(j - goal_column) # |x2-x1| + |y2-y1|
        return total

    def __lt__(self, other):
        return self.f < other.f


def find_position_in_goal(value):
    for i in range(3):
        for j in range(3):
            if GOAL_STATE[i][j] == value:
                return i, j


def find_blank(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                return i, j

 
def swap(state, i1, j1, i2, j2):  # swaps blank tile with another tile
    temp = [[0,0,0],[0,0,0],[0,0,0]]

    for i in range(3):
        for j in range(3):
            temp[i][j] = state[i][j]

    value = temp[i1][j1]
    temp[i1][j1] = temp[i2][j2]
    temp[i2][j2] = value

    new_state = ((temp[0][0], temp[0][1], temp[0][2]),
                 (temp[1][0], temp[1][1], temp[1][2]),
                 (temp[2][0], temp[2][1], temp[2][2]))
    return new_state


def reconstruct_path(node):
    moves = []
    while node.parent is not None:
        moves.append(node.move)
        node = node.parent
    moves.reverse()
    return moves


def a_star(initial_state):
    start = Node(initial_state)

    open_list = []
    heapq.heappush(open_list, start)

    visited = set()

    while open_list:
        current = heapq.heappop(open_list)

        if current.state == GOAL_STATE:
            return current

        if current.state in visited:
            continue
        visited.add(current.state)

        blank_row, blank_column = find_blank(current.state)

        for delta_row, delta_column in MOVES:
            new_row = blank_row + delta_row
            new_column = blank_column + delta_column

            if 0 <= new_row < 3 and 0 <= new_column < 3:
                new_state = swap(current.state, blank_row, blank_column, new_row, new_column)

                if new_state not in visited:
                    child = Node(new_state, current, (delta_row, delta_column), current.g + 1)
                    heapq.heappush(open_list, child)

    return None


def print_state(state):
    for row in state:
        print(row)
    print()


def move_name(move):
    if move == (-1,0): return "Up"
    if move == (1,0):  return "Down"
    if move == (0,-1): return "Left"
    if move == (0,1):  return "Right"
    return str(move)


# Initial State
initial_state = ((0,1,2),
                 (3,5,6),
                 (4,7,8))

goal_node = a_star(initial_state)

if goal_node is None:
    print("No solution found.")
else:
    moves = reconstruct_path(goal_node)

    print("Initial State:")
    print_state(initial_state)

    current_var = initial_state
    for move in moves:
        blank_row, blank_column = find_blank(current_var)
        new_row = blank_row + move[0]
        new_column = blank_column + move[1]
        current_var = swap(current_var, blank_row, blank_column, new_row, new_column)

        print("Move:", move_name(move))
        print_state(current_var)

    print("Total cost (moves):", len(moves))


Initial State:
(0, 1, 2)
(3, 5, 6)
(4, 7, 8)

Move: Right
(1, 0, 2)
(3, 5, 6)
(4, 7, 8)

Move: Down
(1, 5, 2)
(3, 0, 6)
(4, 7, 8)

Move: Left
(1, 5, 2)
(0, 3, 6)
(4, 7, 8)

Move: Down
(1, 5, 2)
(4, 3, 6)
(0, 7, 8)

Move: Right
(1, 5, 2)
(4, 3, 6)
(7, 0, 8)

Move: Right
(1, 5, 2)
(4, 3, 6)
(7, 8, 0)

Move: Up
(1, 5, 2)
(4, 3, 0)
(7, 8, 6)

Move: Left
(1, 5, 2)
(4, 0, 3)
(7, 8, 6)

Move: Up
(1, 0, 2)
(4, 5, 3)
(7, 8, 6)

Move: Right
(1, 2, 0)
(4, 5, 3)
(7, 8, 6)

Move: Down
(1, 2, 3)
(4, 5, 0)
(7, 8, 6)

Move: Down
(1, 2, 3)
(4, 5, 6)
(7, 8, 0)

Total cost (moves): 12


TASK# 02
Does Greedy Best-First Search guarantee shortest path?
No.
Greedy Best-First Search chooses the next city only by lowest heuristic h(n) (closest “looks like” to Bucharest). It ignores the real distance already traveled, so it can take a route that seems closer but is actually longer.

In [22]:
import heapq


graph = {
    'Arad': {'Zerind': 75, 'Sibiu': 140, 'Timisoara': 118},
    'Zerind': {'Arad': 75, 'Oradea': 71},
    'Oradea': {'Zerind': 71, 'Sibiu': 151},
    'Sibiu': {'Arad': 140, 'Oradea': 151, 'Fagaras': 99, 'Rimnicu Vilcea': 80},
    'Timisoara': {'Arad': 118, 'Lugoj': 111},
    'Lugoj': {'Timisoara': 111, 'Mehadia': 70},
    'Mehadia': {'Lugoj': 70, 'Drobeta': 75},
    'Drobeta': {'Mehadia': 75, 'Craiova': 120},
    'Craiova': {'Drobeta': 120, 'Rimnicu Vilcea': 146, 'Pitesti': 138},
    'Rimnicu Vilcea': {'Sibiu': 80, 'Craiova': 146, 'Pitesti': 97},
    'Fagaras': {'Sibiu': 99, 'Bucharest': 211},
    'Pitesti': {'Rimnicu Vilcea': 97, 'Craiova': 138, 'Bucharest': 101},
    'Bucharest': {'Fagaras': 211, 'Pitesti': 101, 'Giurgiu': 90, 'Urziceni': 85},
    'Giurgiu': {'Bucharest': 90},
    'Urziceni': {'Bucharest': 85, 'Hirsova': 98, 'Vaslui': 142},
    'Hirsova': {'Urziceni': 98, 'Eforie': 86},
    'Eforie': {'Hirsova': 86},
    'Vaslui': {'Urziceni': 142, 'Iasi': 92},
    'Iasi': {'Vaslui': 92, 'Neamt': 87},
    'Neamt': {'Iasi': 87}
}


heuristic = {
    'Arad': 366, 'Bucharest': 0, 'Craiova': 160, 'Drobeta': 242, 'Eforie': 161,
    'Fagaras': 176, 'Giurgiu': 77, 'Hirsova': 151, 'Iasi': 226, 'Lugoj': 244,
    'Mehadia': 241, 'Neamt': 234, 'Oradea': 380, 'Pitesti': 100,
    'Rimnicu Vilcea': 193, 'Sibiu': 253, 'Timisoara': 329, 'Urziceni': 80,
    'Vaslui': 199, 'Zerind': 374
}


class Node:
    def __init__(self, city, parent=None, cost=0):
        self.city = city
        self.parent = parent
        self.cost = cost   # total distance travelled so far
        self.h = self.heuristic_function()

    def heuristic_function(self):
        return heuristic[self.city]  # Greedy uses only h(n)

    def __lt__(self, other):
        return self.h < other.h  # priority by smallest heuristic


def reconstruct_path(node):
    path = []
    while node is not None:
        path.append(node.city)
        node = node.parent
    path.reverse()
    return path


def greedy_best_first_search(start, goal):
    open_list = []
    heapq.heappush(open_list, Node(start))

    visited = set()
    visited_order = []

    while open_list:
        current = heapq.heappop(open_list)

        if current.city in visited:
            continue

        visited.add(current.city)
        visited_order.append(current.city)

        if current.city == goal:
            return reconstruct_path(current), current.cost, visited_order

        for neighbor in graph[current.city]:
            if neighbor not in visited:
                new_cost = current.cost + graph[current.city][neighbor]
                heapq.heappush(open_list, Node(neighbor, current, new_cost))

    return None, None, visited_order



start_city = 'Arad'
goal_city = 'Bucharest'

path, total_distance, visited_cities = greedy_best_first_search(start_city, goal_city)

print("Cities visited (in order):")
print(" >>> ".join(visited_cities))

print()

print("Route found by Greedy Best-First Search:")
print(" >>> ".join(path))
print("Total distance traveled:", total_distance)



Cities visited (in order):
Arad >>> Sibiu >>> Fagaras >>> Bucharest

Route found by Greedy Best-First Search:
Arad >>> Sibiu >>> Fagaras >>> Bucharest
Total distance traveled: 450
